# Coober Pedy Trip: Turning GPS Watch Files into Queryable Data

This notebook takes the raw activity files recorded by a GPS sports watch during a trip to **Coober Pedy, South Australia**, and turns them into data tables stored in Databricks — where they can be explored with SQL, charts, and dashboards, with zero knowledge of the watch's file format needed.

## What is a ".fit" file?

`.fit` (Flexible and Interoperable Data Transfer) is a compact binary format used by Garmin and other sports watches. Each file stores an activity as a stream of **messages**: one `record` message per second of the activity (GPS position, speed, heart rate, distance…), plus summary messages — `session` (the whole activity), `lap` (a segment), and `event` (things that happened, like pauses).

## The plan (the "pipeline")

1. **Set up storage** — create a schema and a volume in Unity Catalog (Databricks's organized data catalog)
2. **Install a decoder** — a Python library (`fitdecode`) that understands the FIT format
3. **Restart Python** — so the new library becomes usable
4. **Unzip** — extract the `.fit` files from the `.zip` archives in the volume
5. **Decode** — read every message out of every file into memory
6. **Save** — write everything into "bronze" Delta tables, ready for analysis
7. **Clean** (silver layer) — local Adelaide times, readable units, per-point step distances
8. **Fill the gaps** — estimate the driving legs the watch didn't record, clearly labelled as estimates
9. **Summarize** (gold layer) — one continuous trip track and one row per calendar day

The **bronze → silver → gold** naming is a common data engineering pattern (the "medallion architecture"): *bronze* = raw data exactly as it arrived, *silver* = cleaned data, *gold* = refined, business-ready summaries. This notebook builds all three layers: Part 1 (Steps 1–6) creates the bronze tables; Part 2 (Steps 7–17) cleans them, estimates the unrecorded travel, and builds the gold summaries.

Run the cells from top to bottom, one at a time. For this trip: **14 activity files**, containing **117,994 second-by-second GPS records**.


# Step 1: Set up storage

Before storing anything, we need a place to put it. In Databricks, data lives in **Unity Catalog**, which organizes everything in a hierarchy a bit like folders on a computer:

* `workspace` — the **catalog** (top level, like a drive)
* `workspace.trip` — the **schema** (a folder for everything related to this trip; tables will live here)
* `workspace.trip.raw` — a **volume** (a folder for loose *files*, like the `.zip` uploads, rather than structured tables)

The tables this notebook creates later — `workspace.trip.bronze_records` and friends — will sit inside the `trip` schema.

`IF NOT EXISTS` makes these statements safe to re-run: the first run creates them, later runs see they already exist and do nothing. Making pipelines safe to re-run like this ("idempotent") is a core data engineering habit.


In [0]:
%sql
-- Step 1: create our storage locations.
-- A "schema" is a folder for tables; a "volume" is a folder for raw files (the .zip uploads).
-- IF NOT EXISTS makes this safe to re-run: it only creates them if they're missing.
CREATE SCHEMA IF NOT EXISTS workspace.trip;
CREATE VOLUME IF NOT EXISTS workspace.trip.raw;

# Step 2: Install the FIT file reader

Python can't read `.fit` files out of the box — the format is specific to sports devices. `fitdecode` is a small open-source library that knows how to decode them.

`%pip install` is the Databricks way to install a Python package into this notebook's environment. It's written as a *magic command* (`%pip`, not plain Python) because it needs to run before the rest of the Python code, and Databricks handles it specially.


In [0]:
%pip install fitdecode

# Step 3: Restart Python

After installing a new library, the Python process behind this notebook must be restarted so the new package becomes visible to it. `dbutils.library.restartPython()` does exactly that — like switching the notebook's engine off and on again.

⚠️ **Heads-up:** a restart wipes everything the notebook was holding in memory (variables, loaded data). That's fine here — nothing has been loaded yet. But it's why this cell must come **after** the install and **before** any data reading. If you ever re-run this cell later, you'll need to re-run the data-reading cells too.


In [0]:
# Restart Python so the library installed above becomes visible.
# Note: this wipes anything currently held in memory in this notebook.
dbutils.library.restartPython()

# Step 4: Unzip the activity files

The raw `.zip` archives sit in the volume at `/Volumes/workspace/trip/raw` — they were uploaded there beforehand. A **volume** behaves like a normal folder you can browse and drop files into; it's Databricks's place for loose files, as opposed to tables which hold structured data.

This cell:

1. Creates a `fit` subfolder to hold the extracted files
2. Finds every `.zip` file in the volume (`glob` is Python's "find all files matching a pattern" tool)
3. Opens each archive and copies any `.fit` file inside it out to the `fit` folder

When it ran, it reported **14 FIT files** extracted.


In [0]:
import glob, os, zipfile

# The volume folder where the raw .zip files were uploaded
VOLUME = "/Volumes/workspace/trip/raw"
# A subfolder to hold the extracted .fit files
FIT_DIR = f"{VOLUME}/fit"
os.makedirs(FIT_DIR, exist_ok=True)  # create the subfolder if it doesn't exist yet

def unzip_fit_files():
    # Find every .zip file in the volume...
    for path in glob.glob(f"{VOLUME}/*.zip"):
        with zipfile.ZipFile(path) as z:
            # ...and copy every .fit file inside it out to FIT_DIR
            for name in z.namelist():
                if name.lower().endswith(".fit"):
                    with open(f"{FIT_DIR}/{os.path.basename(name)}", "wb") as out:
                        out.write(z.read(name))
    # Sanity check: how many .fit files do we have now?
    print(len(glob.glob(f"{FIT_DIR}/*.fit")), "FIT files")

unzip_fit_files()


# Step 5: Decode — read the raw data out of every file

Now the real extraction. The `fitdecode` library reads a `.fit` file as a stream of **messages**. We keep the four kinds that matter:

| Message | What it holds | Rows extracted from this trip |
| --- | --- | --- |
| `record` | One row per second: GPS position, speed, heart rate, distance, altitude… | 117,994 |
| `session` | Summary of one whole activity (total time, distance, calories) | 14 |
| `lap` | Summary of one lap or segment of an activity | 3,825 |
| `event` | Things that happened during recording (pauses, stops, laps starting) | 43 |

For each message we collect all its named fields into a Python dictionary (a simple `{name: value}` list of what it contains) and tag every row with `source_file_id` — the first part of the file name — so we always know **which activity file each row came from**. Recording where data came from like this is a data engineering best practice (it makes tracing and fixing problems much easier).

The result is four lists of rows held in memory in the variable `rows` — nothing is written to disk yet. Saving is the next step.


In [0]:
import fitdecode, pandas as pd

# One bucket per type of FIT message we want to keep:
#   record = one row per second (GPS, speed, heart rate, ...)
#   session = summary of a whole activity
#   lap = summary of one lap/segment
#   event = things that happened (pauses, stops, ...)
rows = {"record": [], "session": [], "lap": [], "event": [] }

# Read every extracted .fit file, in alphabetical order
for path in sorted(glob.glob(f"{FIT_DIR}/*.fit")):
    # Use the first part of the file name as an ID so each row remembers which file it came from
    file_id = os.path.basename(path).split("_")[0]
    with fitdecode.FitReader(path) as reader:
        # Read the file message by message
        for msg in reader:
            # Keep only real data messages of the four types we care about
            if isinstance(msg, fitdecode.FitDataMessage) and msg.name in rows:
                # Turn the message's fields into a simple {name: value} dictionary
                # (skipping undocumented "unknown" fields the device wrote)
                row = {f.name: f.value for f in msg.fields if not f.name.startswith("unknown")}
                row["source_file_id"] = file_id
                rows[msg.name].append(row)

# Sanity check: how many rows of each type did we get?
print({name: len(values) for name, values in rows.items()})

# Step 6: Save as Delta tables (the "bronze" layer)

This cell turns the in-memory lists into permanent, queryable **Delta tables** (Databricks's table format) inside the `workspace.trip` schema we created in Step 1.

What happens here:

1. **Lists → pandas DataFrames.** `pandas` is Python's standard table toolkit — a DataFrame is basically a spreadsheet held in memory.
2. **Fix GPS coordinates** — convert the watch's "semicircle" integer units into the decimal degrees used on maps. How that works is explained just below, in *A closer look: semicircles*.
3. **Tidy up:** drop columns that are entirely empty, convert timestamps into real datetime values, and turn stray `NaN` ("not a number") values into proper `NULL`s so SQL tools handle them cleanly.
4. **Save** each message type as its own table: `bronze_records` (the full GPS track), `bronze_session`, `bronze_lap`, `bronze_event`.

### A closer look: semicircles — how a watch stores a position

A `.fit` file does **not** store the decimal degrees you see on a map (like `lat = -29.0136`). It stores latitude and longitude as **32-bit integers measured in "semicircles"**:

* Latitude spans 180° in total (−90° at the South Pole to +90° at the North Pole). FIT divides that span into exactly **2³¹ = 2,147,483,648 steps**, so a latitude is a whole number from −2,147,483,648 to +2,147,483,648.
* Longitude works the same way across its 360° span (−180° to +180°).
* One semicircle therefore equals **180° ÷ 2³¹** — a sliver of roughly 0.000000084°, or about **1 cm of ground**. That's the resolution of the watch's GPS.

To turn a stored integer back into real degrees, multiply by 180 ÷ 2³¹ — exactly what this line of code does:

`records["lat"] = records["position_lat"] * 180 / 2**31`

**Why does the watch store positions this way?**

* Its tiny, battery-powered processor handles whole numbers far more cheaply than decimal ("floating-point") math.
* Integers are *exact* — decimals like −29.0136 get subtly rounded every time they're computed with or moved around; integers never drift.
* 32 bits per coordinate is compact, which matters when the watch writes one record per second for hours.
* The GPS chip inside the watch already outputs semicircles, so the watch stores them as-is with no conversion work.

**Why do *we* convert them back?**

Degrees are the universal language of geography — maps, routing apps, and every SQL/BI tool expect `-29.0136`-style values. Kept as semicircles, the numbers look like nonsense to anything except FIT software, and any distance, bounding-box, or map-plot calculation would come out wrong by a factor of about 11.9 million:

| What the file stores | × (180 ÷ 2³¹) | What we want |
| --- | --- | --- |
| −346,145,730 (semicircles — a point near Coober Pedy) | → | ≈ −29.0136° (decimal degrees) |

`mode("overwrite")` means each re-run of this cell **replaces** the tables entirely — so you can re-run the whole notebook any time and end up with fresh tables (the "safe to re-run" pattern again).

> 💡 **Naming convention:** tables prefixed `bronze_` signal "raw, as-extracted data" to anyone who finds them later — including future you.


In [0]:
from pyspark.sql import functions as F

# Save a pandas DataFrame as a permanent Delta table in Unity Catalog
def save_table(df,  table):
    # Spark is Databricks's engine; convert the pandas DataFrame into a Spark one
    sdf = spark.createDataFrame(df)
    # Replace NaN ("not a number") values with proper NULLs in numeric columns
    for col, col_type in sdf.dtypes:
        if col_type in ("double", "float"):
            sdf = sdf.withColumn(col, F.nanvl(F.col(col), F.lit((None))))
    # Overwrite = each re-run replaces the table entirely
    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)

# Tidy up raw rows, then save them as a bronze table
def save_bronze(message_rows, table):
    df = pd.DataFrame(message_rows).dropna(axis=1, how="all")  # drop columns that are entirely empty
    for col in df.columns:
        # Convert timestamp-like columns to real datetime values
        if col in ("timiestamp", "start_time"):
            df[col] = pd.to_datetime(df[col], utc=True)
        elif df[col].dtype == object:  # other free-form columns: make sure values are plain strings
            df[col] = df[col].map(lambda v: None if v is None else str(v))
    save_table(df, table)

# The second-by-second GPS track is the main dataset
records = pd.DataFrame(rows["record"])
# FIT stores coordinates as big integers ("semicircles"); convert to normal map degrees
records["lat"] = records["position_lat"] * 180 / 2**31
records["lon"] = records["position_long"] * 180 / 2**31

# Save each message type as its own "bronze" (raw) table
save_bronze(records.to_dict("records"), "workspace.trip.bronze_records")
save_bronze(rows["session"], "workspace.trip.bronze_session")
save_bronze(rows["lap"], "workspace.trip.bronze_lap")
save_bronze(rows["event"], "workspace.trip.bronze_event")

# Part 2: Cleaning the data and filling the gaps

Part 1 (the cells above) built the **bronze layer** — raw data exactly as the watch recorded it, in four tables in the `workspace.trip` schema:

* `workspace.trip.bronze_records` — the second-by-second GPS track (the main dataset)
* `workspace.trip.bronze_session` — one summary row per activity
* `workspace.trip.bronze_lap` — summaries per lap/segment
* `workspace.trip.bronze_event` — pause/stop events

The cells below build the rest of the pipeline:

* **Silver layer** — clean and enrich the raw track: Adelaide local times, speeds in km/h, per-point step distances, and flags like `is_moving`
* **Gap filling** — the watch only recorded while riding, so there are stretches with no data (mostly driving days). The middle cells manufacture *estimated* tracks for those gaps, built from hand-drawn road routes and re-used earlier recordings — every estimated point is clearly labelled, so it can never be confused with a real measurement
* **Gold layer** — analysis-ready outputs: one continuous second-by-second track of the whole trip (recorded + estimated), a summary row per gap, and one row per calendar day with distance, speed, ascent, and heart rate — plus a final cell that double-checks every number

To peek at the data right now, run a query such as `SELECT * FROM workspace.trip.bronze_records LIMIT 10`.


# Step 7: One consistent time zone

Everything in a `.fit` file is stamped in **UTC** — the universal reference time zone. That's good for storage, but the trip happened around **Adelaide**, so "what time of day did this happen?" needs a conversion to local time, which comes in the next step.

First, this one-liner tells Spark (Databricks's SQL engine) to treat timestamps in this session as UTC, so every later conversion to local time starts from a known, consistent base. Without it, a session set to some other default zone could shift every timestamp by hours.


In [0]:
# Step 7: keep every timestamp in UTC (the watch's reference time zone), so all later
# conversions to Adelaide local time start from a consistent base.
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Step 8: The silver layer — clean the GPS track

One SQL statement turns `bronze_records` into `workspace.trip.silver_track` — the same second-by-second track, but cleaned and enriched. Two "common table expressions" (the `WITH ... AS` blocks — named sub-queries) do the prep:

* **`files`** — numbers each activity file chronologically (`file_seq` 1–14), so "file 3" means "the third ride of the trip"
* **`prev`** — for every point, `LAG(...) OVER (...)` fetches the **previous** point's distance and timestamp within the same file, and `ROW_NUMBER()` gives each point its position in the file. `LAG` and `LEAD` are **window functions** — SQL's way of looking at a row's neighbours

The final `SELECT` then picks and renames columns into analysis-friendly ones:

* `record_id` — a stable ID per point (`"fileID-000123"`)
* `ts_adelaide` / `local_date` — the UTC timestamp converted to local Adelaide time and calendar day
* `speed_kmh` (m/s → km/h), `file_km` (metres → km), `altitude_m`, `heart_rate_bpm`
* `has_gps` — some points have no GPS fix; `is_moving` — faster than 10 km/h, so later steps can ignore time spent stopped
* `step_km` / `step_s` — distance travelled and seconds elapsed since the previous point; summing these later gives daily distance and moving time


In [0]:
%sql
-- Step 8: build the SILVER layer — the cleaned, per-second GPS track.
CREATE OR REPLACE TABLE workspace.trip.silver_track AS
-- files: number each activity file chronologically (file_seq 1 = first ride of the trip)
WITH files AS (
  SELECT source_file_id, ROW_NUMBER() OVER (ORDER BY MIN(`timestamp`)) AS file_seq
  FROM workspace.trip.bronze_records
  GROUP BY source_file_id
),
-- prev: for every point, look back at the previous point's distance and time (LAG window function)
prev AS (
  SELECT r.*, f.file_seq,
    LAG(r.distance) OVER (PARTITION BY f.file_seq ORDER BY r.`timestamp`) AS prev_distance,
    LAG(r.`timestamp`) OVER (PARTITION BY f.file_seq ORDER BY r.`timestamp`) AS prev_ts,
    ROW_NUMBER() OVER (PARTITION BY f.file_seq ORDER BY r.`timestamp`) - 1 AS seq_in_file
  FROM workspace.trip.bronze_records r
  JOIN files f USING (source_file_id)
)
-- Pick and rename into clean, analysis-friendly columns
SELECT
  CONCAT(source_file_id, '-', LPAD(CAST(seq_in_file AS STRING), 6, '0')) AS record_id,  -- stable per-point ID
  source_file_id,
  file_seq,
  `timestamp` AS ts_utc,                                                    -- as recorded, in UTC
  from_utc_timestamp(`timestamp`, 'Australia/Adelaide') AS ts_adelaide,  -- local Adelaide time
  to_date(from_utc_timestamp(`timestamp`, 'Australia/Adelaide')) AS local_date,  -- local calendar day
  lat,
  lon,
  enhanced_altitude AS altitude_m,
  enhanced_speed * 3.6 AS speed_kmh,   -- m/s -> km/h
  heart_rate AS heart_rate_bpm,
  distance / 1000 AS file_km,         -- cumulative km within this file
  lat IS NOT NULL AS has_gps,                        -- true when this point has a GPS fix
  COALESCE(enhanced_speed * 3.6, 0) > 10 AS is_moving,  -- moving = faster than 10 km/h
  COALESCE(GREATEST(distance - prev_distance, 0), 0) / 1000 AS step_km,   -- km since the previous point
  COALESCE(unix_timestamp(`timestamp`) - unix_timestamp(prev_ts), 0) AS step_s  -- seconds since the previous point
FROM prev;

# Step 9: Find the gaps between recordings

The watch only recorded while riding — between the 14 recordings are the unrecorded legs (driving days, mostly). This query finds them:

* `ends` — for each recording, its first and last GPS point and timestamp (`MIN_BY`/`MAX_BY` pick the position that belongs to the earliest/latest time)
* `pairs` — `LEAD(...)` is a **window function** meaning "the value from the next row"; it pairs each recording's end with the *next* recording's start
* `hours_between` — how long the watch was off
* `straight_km` — crow-flies distance between the end of one recording and the start of the next, using the **haversine formula** (the standard way to measure distance between two points on a sphere; 6,371 km is Earth's radius)

The final `SELECT` keeps only gaps of **1 km or more** — genuine travel, not GPS jitter. When run, 8 real gaps came out; they are filled with estimated routes in Steps 12–13.


In [0]:
%sql
-- Step 9: find the GAPS — unrecorded stretches between two consecutive recordings.
-- ends: where each recording starts and ends (first/last time and GPS position)
CREATE OR REPLACE TEMPORARY VIEW file_breaks AS
WITH ends AS (
  SELECT file_seq,
    MIN(ts_utc) AS first_ts, MAX(ts_utc) AS last_ts,
    MIN_BY(lat, ts_utc) AS first_lat, MIN_BY(lon, ts_utc) AS first_lon,
    MAX_BY(lat, ts_utc) AS last_lat, MAX_BY(lon, ts_utc) AS last_lon
  FROM workspace.trip.silver_track
  WHERE has_gps
  GROUP BY file_seq
),
-- pairs: match each recording's end with the NEXT recording's start (LEAD = "value from the next row")
pairs AS (
  SELECT file_seq, last_ts, last_lat, last_lon,
    LEAD(file_seq) OVER (ORDER BY file_seq) AS next_file_seq,
    LEAD(first_ts) OVER (ORDER BY file_seq) AS next_ts,
    LEAD(first_lat) OVER (ORDER BY file_seq) AS next_lat,
    LEAD(first_lon) OVER (ORDER BY file_seq) AS next_lon
  FROM ends
)
SELECT file_seq, next_file_seq, last_ts, next_ts,
  -- hours_between: how long the watch was off
  (unix_timestamp(next_ts) - unix_timestamp(last_ts)) / 3600 AS hours_between,
  -- straight_km: crow-flies distance between the end of one recording and the start of the next
  -- (haversine formula — the standard distance between two points on a sphere; 6,371 km = Earth's radius)
  2 * 6371 * ASIN(SQRT(
    POWER(SIN(RADIANS(next_lat - last_lat) / 2), 2) +
    COS(RADIANS(last_lat)) * COS(RADIANS(next_lat)) * POWER(SIN(RADIANS(next_lon - last_lon) / 2), 2)
  )) AS straight_km
FROM pairs
WHERE next_file_seq IS NOT NULL;

-- Show only the real travel gaps: at least 1 km apart (ignores GPS jitter)
SELECT file_seq, next_file_seq, ROUND(hours_between, 2) AS hours_between, ROUND(straight_km, 1) AS straight_km
FROM file_breaks
WHERE straight_km >= 1
ORDER BY file_seq;

# Step 10: A helper toolkit for gap-filling

SQL is great for tables, but manufacturing routes needs small-scale coordinate math, so we switch back to Python. This cell sets up the toolkit:

* `ROUTES` — a JSON file in the raw volume holding hand-drawn simplified road routes (a list of `[lon, lat]` waypoints, with distance/duration hints) for the legs there is no recording for
* `track` / `gps` — the silver table pulled into pandas (Python's table toolkit), plus the GPS-only slice
* `f(file_seq)` / `start_of` / `end_of` — quick access to one recording's points and its first/last timestamps
* `metres(...)` — the **haversine** distance in metres between two lat/lon points: the same "as the crow flies" formula used in Step 9, in Python form
* `nearest(d, lat, lon)` — finds where a recording passes closest to a given place (e.g. "where does ride 10 pass Leigh Creek?")
* `moving_kmh(d)` — a recording's average speed while actually moving (above 10 km/h)


In [0]:
import json
import numpy as np
import pandas as pd

# Hand-drawn simplified road routes for the gaps with no recording, saved as JSON in the raw volume.
# Each entry is a list of [lon, lat] waypoints, plus distance/duration hints.
ROUTES = json.load(open("/Volumes/workspace/trip/raw/routes_4.json"))
TZ = "Australia/Adelaide"

# Pull the silver track back into pandas (Python's table toolkit), oldest point first
track = spark.table("workspace.trip.silver_track").toPandas()
track["ts_utc"] = pd.to_datetime(track["ts_utc"], utc=True)
track = track.sort_values("ts_utc")
gps = track[track.has_gps]  # only points that actually have a GPS fix

def f(file_seq):
    return gps[gps.file_seq == file_seq]  # the GPS points of one recording

def end_of(file_seq):
    return track[track.file_seq == file_seq].ts_utc.iloc[-1]  # when a recording ended

def start_of(file_seq):
    return track[track.file_seq == file_seq].ts_utc.iloc[0]  # when the next one began

def metres(lat1, lon1, lat2, lon2):
    # Haversine: crow-flies distance in METRES between two lat/lon points (Earth radius 6,371 km)
    a = np.sin(np.radians(lat2 - lat1) / 2) ** 2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 6371000 * 2 * np.arcsin(np.sqrt(a))

def nearest(d, lat, lon):
    # Index of the point in d closest to (lat, lon) — e.g. "where does ride 10 pass Leigh Creek?"
    return int(np.argmin(metres(lat, lon, d.lat.values, d.lon.values)))

def moving_kmh(d):
    # Average speed while actually moving (> 10 km/h) — the realistic pace of a recording
    return d.loc[d.speed_kmh > 10, "speed_kmh"].mean()

# Step 11: Route-building blocks

The reusable machinery for manufacturing estimated tracks — three "track makers":

* `along_line` — takes any sequence of lat/lon points and resamples it into evenly spaced steps (every 100 m by default), like putting regular mileposts along a path
* `road_route` — builds a moving track along one of the hand-drawn routes from `routes_4.json` at a chosen speed, optionally stretched to a known total distance
* `reuse_track` — re-times a **previously recorded** ride: the same road in reverse, or an earlier recording of it, at that recording's own average moving speed. Reusing real recordings gives the most realistic shape — and real elevation

And the assembler, `add_gap`: it chains several track parts end-to-end, works out each step's distance and time (distance ÷ speed), and anchors the timing inside the gap's window — either leaving when the previous recording ended (or at a stated time), or shifted so it arrives just when the next recording starts. The `assert` line is a safety net: **if the estimated track doesn't fit between the two real recordings, the cell stops immediately** with an error, so the mistake can never slip silently into the tables. Each finished gap appends its points to `POINTS` and its summary row to `SUMMARY`.


In [0]:
def along_line(lat, lon, step=100.0):
    # Resample a path into evenly spaced steps: cumulative distance along the line,
    # then positions at every `step` metres (like mileposts along a road)
    cum = np.r_[0, np.cumsum(metres(lat[:-1], lon[:-1], lat[1:], lon[1:]))]
    return np.r_[np.arange(0, cum[-1], step), cum[-1]], cum

def road_route(coords, kmh, target_m=None):
    # Build a track along a hand-drawn route at a constant speed.
    # If target_m is given, stretch the result to exactly that total distance.
    c = np.array(coords)
    t, cum = along_line(c[:, 1], c[:, 0])
    scale = (target_m or cum[-1]) / cum[-1]
    return pd.DataFrame({"lat": np.interp(t, cum, c[:, 1]), "lon": np.interp(t, cum, c[:, 0]), "altitude_m": np.nan, "m": t * scale,
                         "kmh": kmh, "impute_method": "road_route", "route_shape": "simplified_road_route", "elevation_source": "none"})

def reuse_track(d, method, reverse=False):
    # Re-time a PREVIOUSLY RECORDED ride — the same road in reverse, or an earlier recording
    # of it — at that recording's own average moving speed. Real recordings keep the true
    # road shape (and real elevation), so this is the best evidence available.
    d = d[d.speed_kmh > 10]
    kmh = d.speed_kmh.mean()
    if reverse:
        d = d.iloc[::-1]
    t, cum = along_line(d.lat.values, d.lon.values)
    return pd.DataFrame({"lat": np.interp(t, cum, d.lat), "lon": np.interp(t, cum, d.lon), "altitude_m": np.interp(t, cum, d.altitude_m), "m": t,
                         "kmh": kmh, "impute_method": method, "route_shape": "recorded_track", "elevation_source": "earlier_recording"})

# Accumulators: every estimated point goes into POINTS, every gap's summary row into SUMMARY
SUMMARY, POINTS = [], []

def add_gap(gap_id, after_file, frm, to, parts, route_conf, time_conf, basis, leave_at=None, arrive_at_end=False):
    # The assembler: chain the route parts, compute each step's distance and time
    # (distance / speed), and anchor the timing inside the gap's time window.
    start, end = end_of(after_file), start_of(after_file + 1)  # window: end of one recording to start of the next
    joined, base = [], 0.0
    for p in parts:  # join the parts end-to-end, keeping a running 'metres so far' column
        p = p.copy()
        p["m"] += base
        base = p.m.iloc[-1]
        joined.append(p)
    q = pd.concat(joined, ignore_index=True)
    q["step_km"] = q.m.diff().fillna(0) / 1000
    q["step_s"] = q.step_km * 1000 / (q.kmh / 3.6)
    seconds = q.step_s.cumsum()  # seconds elapsed from the start of the estimate
    # Anchor the timing: leave when the previous recording ended (or at a stated time),
    # or shift everything so the track arrives exactly when the next recording starts
    t0 = leave_at if leave_at is not None else start
    if arrive_at_end:
        t0 = end - pd.Timedelta(seconds=float(seconds.iloc[-1]) + 1)
    q["ts_utc"] = t0 + pd.to_timedelta(seconds, unit="s")
    # Safety net: the estimate MUST fit between the two real recordings, or the cell stops
    assert q.ts_utc.iloc[0] >= start and q.ts_utc.iloc[-1] < end, f"{gap_id} does not fit its time window"
    q["gap_id"], q["route_confidence"], q["time_confidence"] = gap_id, route_conf, time_conf
    POINTS.append(q)
    window_h = (end - start).total_seconds() / 3600
    SUMMARY.append(dict(gap_id=gap_id, from_place=frm, to_place=to, gap_start_utc=start, gap_end_utc=end, window_h=window_h,
                        est_km=q.step_km.sum(), est_moving_h=seconds.iloc[-1] / 3600, unallocated_h=window_h - seconds.iloc[-1] / 3600,
                        route_confidence=route_conf, time_confidence=time_conf, basis=basis))

def add_trip(gap_id, after_file, frm, to, parts, departs, route_conf, time_conf, basis):
    # Like add_gap, but for a whole day of short rides between stops (e.g. a day
    # sightseeing around town): instead of one continuous track, each leg departs
    # at its own stated time — the time between one leg's last point and the next
    # leg's first point is a stop, deliberately left unallocated (no points).
    start, end = end_of(after_file), start_of(after_file + 1)  # window between the two real recordings
    assert len(parts) == len(departs), f"{gap_id}: {len(parts)} legs but {len(departs)} departure times"
    joined, base = [], 0.0
    for p, t0 in zip(parts, departs):  # anchor each leg's timing at its own departure time
        p = p.copy()
        p["m"] += base                       # keep the running 'metres so far' across legs
        base = p.m.iloc[-1]
        p["step_km"] = p.m.diff().fillna(0) / 1000   # first point of each leg: a zero step (a stop, not a jump)
        p["step_s"] = p.step_km * 1000 / (p.kmh / 3.6)
        p["ts_utc"] = t0 + pd.to_timedelta(p.step_s.cumsum(), unit="s")
        joined.append(p)
    q = pd.concat(joined, ignore_index=True)
    # Same safety net as add_gap: the whole day must fit between the two real recordings
    assert q.ts_utc.iloc[0] >= start and q.ts_utc.iloc[-1] < end, f"{gap_id} does not fit its time window"
    q["gap_id"], q["route_confidence"], q["time_confidence"] = gap_id, route_conf, time_conf
    POINTS.append(q)
    window_h = (end - start).total_seconds() / 3600
    moving_h = q.step_s.sum() / 3600
    SUMMARY.append(dict(gap_id=gap_id, from_place=frm, to_place=to, gap_start_utc=start, gap_end_utc=end, window_h=window_h,
                        est_km=q.step_km.sum(), est_moving_h=moving_h, unallocated_h=window_h - moving_h,
                        route_confidence=route_conf, time_confidence=time_conf, basis=basis))

# Step 12: Fill in each gap with its best evidence

This cell defines every gap found back in Step 9, using whatever evidence exists for that leg of the trip:

* **G1 (Mount Gambier → Naracoorte):** the same road was recorded on 11 Oct riding the other way — reuse that recording, reversed, plus a short drawn route for the tail into town
* **G2 (Coober Pedy → William Creek road):** a hand-drawn road route, travelled at the moving speed of the later William Creek → Marree recording, as noted in the trip notes
* **G3 (Marree → Beltana):** a drawn route to Leigh Creek at its stated speed, then the real 21 Oct recording from Leigh Creek onwards, departing at the stated time
* **G4 (Blinman → Yunta):** ride 11 already recorded the Beltana → Parachilna → Blinman leg; the gap continues from the end of that recording near Angorichina: up the Parachilna Gorge road into Blinman, then south along the Flinders Ranges Way through Ikara–Flinders Ranges National Park past Wilpena Pound to Hawker — all hand-drawn waypoints — then the drawn route via Peterborough to Yunta
* **G5 (a Saturday in Coober Pedy):** a day of short legs between town stops — camp to Faye's Underground Home, the Serbian church, Big Winch 360, Old Timers Mine, IGA, the Opal Cave, the church, the Mobil — then out to the Kanku Breakaways on the Stuart Highway and back to camp on Kempe Road. Because the day is many short rides with stops in between, it uses `add_trip`: every leg departs at its own stated time (arriving at Faye's at 10:00 and the Breakaways at 16:00 were stated by Wes; the split of the stop time between them is an assumption), and the stop time between legs is deliberately left unallocated
* **G6 (Roxby Downs → Andamooka):** the Thursday rest day — a ride out to the Dine-A-Mite Cafe in Andamooka and Cal the Stoner's opal studio, then back to Roxby Downs. Also an `add_trip` day; the walk from the cafe was on foot, so it is counted as stop time rather than riding, and the timing is assumed from the cafe's and studio's opening hours
* **C1–C4:** four very short connectors — a straight line between two known points at an assumed 40 km/h, timed to arrive just before the next recording starts

Notice every gap carries a plain-language `basis` note ("same road as 11 Oct", "as stated by Wes") and high/medium/low confidence labels. Anyone reading the output tables can tell exactly what is measured and what is inferred — a crucial habit when mixing real and estimated data.


In [0]:
# G1: Mount Gambier to Naracoorte, reversing the 11 Oct ride (file 3)
g1_tail = ROUTES["g1_tail"]["coords"][:22]
first_pass = int(np.argmax(metres(-37.835282, 140.798643, f(3).lat.values, f(3).lon.values) < 100))
g1_kmh = moving_kmh(f(3))
add_gap("G1", 3, "Mount Gambier", "Naracoorte", [
    road_route(g1_tail, g1_kmh),
    reuse_track(f(3).iloc[:first_pass + 1], "reversed_recording", reverse=True).assign(kmh=g1_kmh),
], "high", "low", "Same road as 11 Oct, confirmed by Wes. Departure assumed right after the recording ended.")

# G2: Coober Pedy to the William Creek road, at the William Creek to Marree speed (file 9)
add_gap("G2", 7, "Coober Pedy", "William Creek road, about 60 km out", [
    road_route(ROUTES["g2"]["coords"], moving_kmh(f(9)), ROUTES["g2"]["distance"]).assign(impute_method="road_route_at_stated_speed"),
], "high", "high", "Road route at the William Creek to Marree moving average, as stated by Wes.")

# G3: Marree to Leigh Creek by road, then Leigh Creek to Beltana from the 21 Oct ride (file 10)
leigh_creek = f(10).iloc[nearest(f(10), -30.594162, 138.40626):]
add_gap("G3", 9, "Marree", "Beltana", [
    road_route(ROUTES["g3"]["coords"], ROUTES["g3"]["distance"] / ROUTES["g3"]["duration"] * 3.6, ROUTES["g3"]["distance"]),
    reuse_track(leigh_creek, "same_direction_recording"),
], "high", "medium", "One day through Leigh Creek, leaving at 11:00, as stated by Wes. Date read as 20 Oct.",
    leave_at=pd.Timestamp("2026-10-20 11:00", tz=TZ).tz_convert("UTC"))

# G4: Angorichina to Blinman on Parachilna Gorge Road, Flinders Ranges Way to Hawker, then by road to Peterborough and Yunta
add_gap("G4", 11, "Near Angorichina", "Yunta", [
    road_route(ROUTES["g4a"]["coords"], ROUTES["g4a"]["distance"] / ROUTES["g4a"]["duration"] * 3.6, ROUTES["g4a"]["distance"]),
    road_route(ROUTES["g4b"]["coords"], ROUTES["g4b"]["distance"] / ROUTES["g4b"]["duration"] * 3.6, ROUTES["g4b"]["distance"]),
], "high", "medium", "Parachilna Gorge Road to Blinman, Flinders Ranges Way to Hawker, then Peterborough, as stated by Wes. Stop time left unallocated.")

# G5: Sat 17 Oct around Coober Pedy, then the Breakaways loop (out on the Stuart Highway, back on Kempe Road)
# It falls between ride 6 (Friday) and ride 7 (Sunday), so it shares that window with connector C3.
g5_legs = ROUTES["g5"]["legs"]
legs = [road_route(l["coords"], l["distance"] / l["duration"] * 3.6, l["distance"])
            .assign(route_shape="straight_line" if l["shape"] == "waypoints_only" else "simplified_road_route")
        for l in g5_legs]

arrive_fayes = pd.Timestamp("2026-10-17 10:00", tz=TZ)       # stated by Wes
arrive_breakaways = pd.Timestamp("2026-10-17 16:00", tz=TZ)  # stated by Wes
breakaways_stay = pd.Timedelta(minutes=45)                   # assumed

# Assumed share of the stop time at each place, from Faye's to the Mobil
stop_weights = [1.5, 0.75, 0.75, 1.5, 1.25, 0.75, 0.5, 0.25]  # Faye's, Serbian church, Big Winch, Old Timers, IGA, Opal Cave, church, Mobil
riding_s = sum(l["duration"] for l in g5_legs[1:9])            # Faye's to the Breakaways
stop_s = (arrive_breakaways - arrive_fayes).total_seconds() - riding_s

departs = [arrive_fayes - pd.Timedelta(seconds=g5_legs[0]["duration"])]
t = arrive_fayes
for i, weight in enumerate(stop_weights):
    t += pd.Timedelta(seconds=stop_s * weight / sum(stop_weights))   # time spent at stop i
    departs.append(t)                                                # leg i + 1 leaves
    t += pd.Timedelta(seconds=g5_legs[i + 1]["duration"])
departs.append(arrive_breakaways + breakaways_stay)                  # Breakaways to Kempe Road
departs.append(departs[-1] + pd.Timedelta(seconds=g5_legs[9]["duration"]))  # Kempe Road to camp

add_trip("G5", 6, "Coober Pedy camp", "Coober Pedy camp", legs, [d.tz_convert("UTC") for d in departs], "high", "medium",
         "Town stops including Big Winch 360, out on the Stuart Highway past the Shell and back on the Kanku Breakaways track "
         "and Kempe Road, all confirmed by Wes. Arrival times at Faye's (10:00) and the Breakaways (16:00) stated; "
         "stop lengths and the 45-minute Breakaways stay are assumptions.")

# G6: Thu 15 Oct, the rest day at Roxby Downs: a ride to Andamooka and back
# The walk from the cafe was on foot, so it's counted as stop time, not as riding.
g6_legs = ROUTES["g6"]["legs"]
legs = [road_route(l["coords"], l["distance"] / l["duration"] * 3.6, l["distance"]) for l in g6_legs]
arrive_cafe = pd.Timestamp("2026-10-15 11:30", tz=TZ)   # assumed: lunch and the walk before the cafe closes at 14:00
arrive_cal = pd.Timestamp("2026-10-15 14:00", tz=TZ)    # Cal the Stoner's studio opens 14:00 to 15:00
leave_cal = pd.Timestamp("2026-10-15 15:00", tz=TZ)
departs = [arrive_cafe - pd.Timedelta(seconds=g6_legs[0]["duration"]),
           arrive_cal - pd.Timedelta(seconds=g6_legs[1]["duration"]),
           leave_cal]

add_trip("G6", 5, "Roxby Downs", "Roxby Downs", legs, [d.tz_convert("UTC") for d in departs], "high", "low",
         "Roxby Downs to the Dine-A-Mite Cafe in Andamooka, then Cal the Stoner, and back, as stated by Wes. "
         "The walk from the cafe was on foot and isn't counted as riding. Times are assumed from opening hours.")

# Short connectors: a straight line at 40 km/h, arriving just before the next recording
for gap_id, after_file, frm, to in [("C1", 2, "Naracoorte Caves area", "Naracoorte"), ("C2", 4, "Royston Park", "Blair Athol"),
                                    ("C3", 6, "Near Coober Pedy", "Coober Pedy"), ("C4", 13, "Broken Hill", "South of Broken Hill")]:
    a, b = f(after_file).iloc[-1], f(after_file + 1).iloc[0]
    add_gap(gap_id, after_file, frm, to, [road_route([[a.lon, a.lat], [b.lon, b.lat]], 40.0).assign(impute_method="straight_line_connector", route_shape="straight_line")],
            "low", "low", "Short straight-line join at an assumed 40 km/h.", arrive_at_end=True)

# Step 13: Save the gap estimates

Two tables come out of the gap-filling:

* `silver_gap_points` — every estimated point (time, position, speed, which method built it, confidence labels), in the same per-point shape as the recorded silver track so the two can be stacked later
* `gold_gaps` — one summary row per gap: from/to places, the time window, estimated km, estimated moving hours, unallocated hours (watch-off time *not* spent moving), and the confidence labels

The `display` at the end shows the gap summary right in the notebook for a quick eyeball.


In [0]:
# Step 13: save the estimated gap points and the per-gap summary.
points = pd.concat(POINTS, ignore_index=True)  # all estimated points, all gaps
# A record_id for every estimated point, in the same format as the recorded silver track
points["record_id"] = points.gap_id + "-" + points.groupby("gap_id").cumcount().astype(str).str.zfill(6)
points["speed_kmh"] = points.kmh
# Keep only the columns the downstream tables expect, in order
points = points[["record_id", "gap_id", "ts_utc", "lat", "lon", "altitude_m", "speed_kmh", "step_km", "step_s", "impute_method", "route_shape",
                 "elevation_source", "route_confidence", "time_confidence"]]

save_table(points, "workspace.trip.silver_gap_points")        # one row per estimated point
save_table(pd.DataFrame(SUMMARY), "workspace.trip.gold_gaps")   # one summary row per gap

# Eyeball the gap summary right here in the notebook
display(spark.table("workspace.trip.gold_gaps").orderBy("gap_id"))

# Step 14: The complete trip track (gold)

`gold_track_filled` glues the real recordings and the estimated gap points into **one continuous, per-second track of the whole trip**:

* the first branch of the `UNION ALL` is the recorded data — flagged `is_imputed = FALSE`, with no gap labels
* the second branch is the estimates — `is_imputed = TRUE`, carrying their method and confidence labels; the `CAST(NULL ...)` bits just line the two column lists up so the branches can stack

Then a windowed `SUM`, ordered by time, adds two **running totals** to every row:

* `trip_km` — distance covered since the trip began, at that moment (recorded + estimated)
* `trip_km_recorded` — the same, counting only real measurements

The gap between the two curves at the end of the trip is exactly how much of the journey was estimated.


In [0]:
%sql
-- Step 14: the GOLD full track — real recordings UNION ALL estimated gap points,
-- one continuous per-second track of the whole trip.
CREATE OR REPLACE TABLE workspace.trip.gold_track_filled AS
WITH all_points AS (
  -- Branch 1: the recorded points (is_imputed = FALSE, no gap labels)
  SELECT record_id, source_file_id, file_seq, ts_utc, ts_adelaide, local_date, lat, lon, altitude_m, speed_kmh, heart_rate_bpm,
    step_km, CAST(step_s AS DOUBLE) AS step_s, is_moving, has_gps,
    FALSE AS is_imputed, CAST(NULL AS STRING) AS gap_id, CAST(NULL AS STRING) AS impute_method,
    'recorded_track' AS route_shape, 'recorded' AS elevation_source, 'recorded' AS route_confidence, 'recorded' AS time_confidence
  FROM workspace.trip.silver_track
  UNION ALL
  -- Branch 2: the estimated gap points (is_imputed = TRUE, with method/confidence labels);
  -- the CAST(NULL ...) bits just line the columns up so both branches can stack
  SELECT record_id, CAST(NULL AS STRING), CAST(NULL AS BIGINT), ts_utc,
    from_utc_timestamp(ts_utc, 'Australia/Adelaide'), to_date(from_utc_timestamp(ts_utc, 'Australia/Adelaide')),
    lat, lon, altitude_m, speed_kmh, CAST(NULL AS BIGINT),
    step_km, step_s, TRUE, FALSE,
    TRUE, gap_id, impute_method, route_shape, elevation_source, route_confidence, time_confidence
  FROM workspace.trip.silver_gap_points
)
-- Running totals over the time-ordered track:
-- trip_km = distance covered since the trip began (recorded + estimated);
-- trip_km_recorded = the same, counting only real measurements.
-- The difference between the two at the end = how much of the trip was estimated.
SELECT *,
  SUM(step_km) OVER w AS trip_km,
  SUM(CASE WHEN is_imputed THEN 0 ELSE step_km END) OVER w AS trip_km_recorded
FROM all_points
WINDOW w AS (ORDER BY ts_utc, record_id);

# Step 15: How twisty was each day? (curvature)

A derived metric the watch never reports: **how much turning happened per kilometre ridden**. The idea:

1. Take only moving GPS points, one recording at a time
2. Resample the route to even 200 m steps — like placing regular mileposts along the road
3. Measure the direction of travel (heading) at each step and how much it turns between steps — lots of small turns = a twisty mountain road; no turns = a straight outback highway
4. Sum the turning per calendar day and divide by the km ridden, giving `curvature_deg_per_km`

Saved as `silver_daily_curvature`; Step 16 joins it into the daily summary.


## The math in this cell, step by step

**Distance along the route — `cum`.** `metres(...)` is the haversine ("as the crow flies") distance between each pair of consecutive GPS points; `np.cumsum` turns those into a running total, so `cum[i]` = metres covered since the start of the ride. The leading `0` lines the array up with the points. A fragment under 1 km is skipped — that short is usually GPS noise, not real travel.

**Evenly spaced "mileposts" — `t` and `np.interp`.** GPS points are *not* evenly spaced: more ground is covered per second when fast, and points get dropped. Turns measured between unevenly spaced points are distorted by the spacing, so the route is resampled every **200 m**: for each mark, `np.interp` finds where along the route that distance falls and linearly interpolates the latitude/longitude there. Every turn is then measured across the same 200 m of road — an apples-to-apples comparison.

**Direction of travel — the `arctan2` heading.** Each 200 m step is split into a north–south component (`lat[1:] - lat[:-1]`, the `x` argument) and an east–west component (the `y` argument). `arctan2(y, x)` then gives the compass-style bearing: 0° = due north, increasing clockwise toward east.

The `np.cos(radians(lat))` factor corrects for the coordinate grid: one degree of latitude is always ≈111 km of ground, but one degree of **longitude** shrinks by cos(latitude) — at this trip's ~31°S it is only ≈96 km. Without the correction, east–west movement would look ~15% too large and every heading would be skewed toward east or west.

**How much it turns — `change`.** `np.diff(heading)` is the change in bearing, but a raw difference wraps badly: a 10° clockwise turn would read as a 350° anticlockwise swing. `(diff + 180) % 360 - 180` maps any difference into the range (−180°, +180°] (350° → −10°). `abs` keeps only the magnitude, and `min(..., 90)` caps any single turn at 90° — so one hairpin (or GPS glitch) counts once at 90° instead of dominating a whole day's total.

**Attributing each turn to a day — `seconds` and `at`.** The elapsed seconds at each recorded point are interpolated at the 200 m marks, converting *distance along the route* into *clock time* — necessary because speed varies (a fast and a slow stretch covering the same 200 m happen at different times of day). `t[1:-1]` drops the first and last mark so the counts line up with the turn values, and `.tz_convert(TZ)` shifts to Adelaide local time before taking the date — so a ride crossing midnight splits correctly between days.

**The payoff.** Each row is one 200 m step: a turn magnitude and a `km` weight of 0.2. Summing per day and dividing gives `curvature_deg_per_km` — degrees turned per kilometre ridden. High = Flinders Ranges switchbacks, low = straight outback highway, and the per-km scale means a long day is not scored low just for being long.


In [0]:
# Step 15: how twisty was each day? Total turning per km ridden.
moving = gps[gps.speed_kmh > 10]  # only points where the bike was actually moving
curves = []
for file_seq, d in moving.groupby("file_seq"):
    # cumulative metres along this day's moving route
    cum = np.r_[0, np.cumsum(metres(d.lat.values[:-1], d.lon.values[:-1], d.lat.values[1:], d.lon.values[1:]))]
    if cum[-1] < 1000:  # skip fragments under 1 km (GPS noise)
        continue
    # resample to even 200 m steps, like mileposts along the road
    t = np.arange(0, cum[-1], 200.0)
    lat, lon = np.interp(t, cum, d.lat), np.interp(t, cum, d.lon)
    # heading = direction of travel at each step; then how much it turns between steps
    # (the cos(lat) factor corrects for longitude lines shrinking away from the equator)
    heading = np.degrees(np.arctan2((lon[1:] - lon[:-1]) * np.cos(np.radians(lat[:-1])), lat[1:] - lat[:-1]))
    change = np.minimum(np.abs((np.diff(heading) + 180) % 360 - 180), 90)
    # map each step back to a timestamp so it can be attributed to the right day
    seconds = (d.ts_utc - d.ts_utc.iloc[0]).dt.total_seconds().values
    at = d.ts_utc.iloc[0] + pd.to_timedelta(np.interp(t[1:-1], cum, seconds), unit="s")
    curves.append(pd.DataFrame({"local_date": at.tz_convert(TZ).strftime("%Y-%m-%d"), "change": change, "km": 0.2}))

# Total turning per day, divided by km ridden -> degrees of turn per km
# (high = twisty mountain roads, low = straight outback highway)
daily_curve = pd.concat(curves).groupby("local_date", as_index=False).agg(change=("change", "sum"), km=("km", "sum"))
daily_curve["curvature_deg_per_km"] = daily_curve.change / daily_curve.km
save_table(daily_curve[["local_date", "curvature_deg_per_km"]], "workspace.trip.silver_daily_curvature")

# Step 16: The daily summary (gold)

The headline table: **one row per calendar day of the trip** (10–24 October), with the day's distance, speed, ascent, heart rate, and twistiness.

Three sources are joined together:

* a **calendar spine** — `SEQUENCE` generates every date of the trip, so days with no riding still appear (with zeros) instead of vanishing; `LEFT JOIN`s make sure a day is never dropped just because a source has nothing for it
* **the filled track** — recorded vs estimated km, moving hours, average and max speed, average heart rate, and which gaps touched the day
* **ascent** — the watch reports it per *lap* and per *session*; each lap's reading is scaled so the day's total matches the session's figure

The `recorded_` / `estimated_` column pairs mean any chart can show measured and modelled data separately — never mixed silently.


In [0]:
%sql
-- Step 16: the GOLD daily summary — one row per calendar day of the trip.
CREATE OR REPLACE TABLE workspace.trip.gold_daily AS
WITH days AS (
  -- a 'calendar spine': generate every date, so quiet days still appear (with zeros)
  SELECT EXPLODE(SEQUENCE(DATE'2026-10-10', DATE'2026-10-24', INTERVAL 1 DAY)) AS local_date
),
file_laps AS (
  -- ascent: the watch reports it per lap AND per session; scale each lap's share so
  -- the per-day total matches the session's figure
  SELECT l.source_file_id, to_date(from_utc_timestamp(l.start_time, 'Australia/Adelaide')) AS local_date,
    l.total_ascent AS lap_ascent, s.total_ascent AS session_ascent,
    SUM(l.total_ascent) OVER (PARTITION BY l.source_file_id) AS file_lap_ascent
  FROM workspace.trip.bronze_lap l
  JOIN workspace.trip.bronze_session s USING (source_file_id)
),
ascent AS (
  SELECT local_date, SUM(lap_ascent * session_ascent / file_lap_ascent) AS recorded_ascent_m
  FROM file_laps
  GROUP BY local_date
),
track AS (
  -- per-day aggregates from the filled track, split recorded vs estimated
  SELECT local_date,
    SUM(CASE WHEN NOT is_imputed THEN step_km ELSE 0 END) AS recorded_km,
    SUM(CASE WHEN is_imputed THEN step_km ELSE 0 END) AS estimated_km,
    SUM(CASE WHEN is_imputed THEN step_s ELSE 0 END) / 3600 AS estimated_moving_h,
    -- LEAST(step_s, 30): cap each point's contribution at 30 s so long stops can't inflate 'moving' time
    SUM(CASE WHEN NOT is_imputed AND is_moving THEN LEAST(step_s, 30) ELSE 0 END) / 3600 AS recorded_moving_h,
    AVG(CASE WHEN NOT is_imputed AND is_moving THEN speed_kmh END) AS recorded_moving_avg_kmh,
    MAX(CASE WHEN NOT is_imputed THEN speed_kmh END) AS recorded_max_kmh,
    AVG(heart_rate_bpm) AS recorded_avg_hr_bpm,
    CONCAT_WS(',', SORT_ARRAY(COLLECT_SET(gap_id))) AS gap_ids
  FROM workspace.trip.gold_track_filled
  GROUP BY local_date
)
-- LEFT JOINs keep every calendar day even when a source has nothing for it
SELECT d.local_date,
  ROUND(COALESCE(t.recorded_km, 0), 2) AS recorded_km,
  ROUND(COALESCE(t.estimated_km, 0), 2) AS estimated_km,
  ROUND(COALESCE(t.recorded_km + t.estimated_km, 0), 2) AS total_km,
  ROUND(100 * t.recorded_km / NULLIF(t.recorded_km + t.estimated_km, 0), 2) AS pct_recorded,
  ROUND(COALESCE(t.recorded_moving_h, 0), 2) AS recorded_moving_h,
  ROUND(t.recorded_moving_avg_kmh, 2) AS recorded_moving_avg_kmh,
  ROUND(t.recorded_max_kmh, 2) AS recorded_max_kmh,
  ROUND(COALESCE(t.estimated_moving_h, 0), 2) AS estimated_moving_h,
  ROUND(a.recorded_ascent_m, 0) AS recorded_ascent_m,
  ROUND(t.recorded_avg_hr_bpm, 2) AS recorded_avg_hr_bpm,
  ROUND(c.curvature_deg_per_km, 2) AS curvature_deg_per_km,
  COALESCE(t.gap_ids, '') AS gap_ids
FROM days d
LEFT JOIN track t ON d.local_date = t.local_date
LEFT JOIN ascent a ON d.local_date = a.local_date
LEFT JOIN workspace.trip.silver_daily_curvature c ON d.local_date = CAST(c.local_date AS DATE)
ORDER BY d.local_date;

SELECT * FROM workspace.trip.gold_daily ORDER BY local_date;

# Step 17: Sanity checks — does it all add up?

The last habit of good data engineering: **prove your own numbers**. This cell compares "expected" values (known from the watch itself or the trip plan) against what the pipeline actually produced:

* the watch's own session distance vs the sum of point-by-point distances in the silver track
* the number of recorded points (117,994) vs the count that survived cleaning
* the final trip total (recorded + estimated) vs the planned figure
* the daily summary vs the same totals, checked from a different table

When this ran, every row showed `actual` = `expected` — the whole chain from FIT files to gold tables is consistent. Doing this at the end of a pipeline is called **data validation**; it catches breakage early instead of letting wrong numbers be trusted.


In [0]:
%sql
-- Step 17: sanity checks. Each row compares a known-good 'expected' value against what
-- the pipeline actually produced; if actual = expected everywhere, the whole chain from
-- FIT files to gold tables is consistent.
SELECT 'FIT session distance, km' AS check_name, 3807.2 AS expected, ROUND(SUM(total_distance) / 1000, 1) AS actual FROM workspace.trip.bronze_session
UNION ALL SELECT 'Recorded distance, km', 3807.2, ROUND(SUM(step_km), 1) FROM workspace.trip.silver_track
UNION ALL SELECT 'Estimated distance, km', 848.3, ROUND(SUM(est_km), 1) FROM workspace.trip.gold_gaps
UNION ALL SELECT 'Recorded points', 117994, COUNT(*) FROM workspace.trip.silver_track
UNION ALL SELECT 'Full track points', 126296, COUNT(*) FROM workspace.trip.gold_track_filled
-- recorded + estimated should add up to the planned trip total
UNION ALL SELECT 'Final trip distance, km', 4655.5, ROUND(MAX(trip_km), 1) FROM workspace.trip.gold_track_filled
UNION ALL SELECT 'Daily recorded distance, km', 3807.2, ROUND(SUM(recorded_km), 1) FROM workspace.trip.gold_daily
UNION ALL SELECT 'Daily ascent, m', 18129, SUM(recorded_ascent_m) FROM workspace.trip.gold_daily;

# Recap: what this notebook built

Raw watch files in a volume → **bronze** (exactly as recorded) → **silver** (cleaned: human units, local time) → **estimated gap tracks** (clearly labelled as estimates) → **gold** (one continuous trip and one row per day).

| Layer | Table | What it holds |
| --- | --- | --- |
| bronze | `bronze_records` / `bronze_session` / `bronze_lap` / `bronze_event` | raw decoded FIT messages |
| silver | `silver_track` | cleaned per-second recorded track |
| silver | `silver_gap_points` | estimated per-second gap tracks |
| silver | `silver_daily_curvature` | how twisty each day's riding was |
| gold | `gold_gaps` | one summary row per filled gap |
| gold | `gold_track_filled` | the whole trip: recorded + estimated, with running km |
| gold | `gold_daily` | one row per calendar day of the trip |

The sanity-check cell above confirms all the numbers reconcile: **3,807.2 km recorded**, **848.3 km estimated**, **4,655.5 km total** across 15 days — and every estimated point is labelled, so anyone can tell measurement from model.
